# Duka Electronics — H1 2026 Sales Exploration

## Executive Summary
- **Nairobi** is the top branch with KES 26,106,100 (36.4% of the KES 71,792,500 total revenue across 1,000 orders).
- **Laptop** is the top product by revenue at KES 41,535,000, while **Tablet** leads on units with 691 sold.
- Weekend orders are 27.7% of orders (277), averaging KES 71,972 per order vs KES 71,724 on weekdays.
- **April** was the best month (KES 13,518,300) and **June** the worst (KES 10,903,400), a 24.0% gap.
- 83 Premium online orders from Nairobi and Mombasa generated KES 10,218,000, i.e. 14.2% of revenue.

## Task 1 · Load & first-five-minutes inspection

In [1]:
import pandas as pd, numpy as np
pd.options.display.float_format = "{:,.2f}".format
df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
print("shape:", df.shape); print(df.dtypes); print("\nmissing values:\n", df.isna().sum())
print("\ndate range:", df.order_date.min().date(), "to", df.order_date.max().date())
for c in ["branch","product","channel"]: print("\n", df[c].value_counts().to_dict())
print("duplicate order_id:", df.order_id.duplicated().sum())
df.describe()

shape: (1000, 7)
order_id               int64
order_date    datetime64[us]
branch                   str
product                  str
quantity               int64
channel                  str
unit_price             int64
dtype: object

missing values:
 order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

date range: 2026-01-01 to 2026-06-29

 {'Nairobi': 376, 'Mombasa': 186, 'Nakuru': 162, 'Kisumu': 159, 'Eldoret': 117}

 {'Laptop': 220, 'Tablet': 215, 'Phone': 194, 'Headphones': 187, 'Charger': 184}

 {'In-store': 579, 'Online': 320, 'Phone order': 101}
duplicate order_id: 0


,order_id,order_date,quantity,unit_price
count,"1,000.00",1000,"1,000.00","1,000.00"
mean,"1,500.50",2026-03-30 08:44:09.600000,3.01,"23,918.30"
min,"1,001.00",2026-01-01 00:00:00,1.00,"1,200.00"
25%,"1,250.75",2026-02-14 00:00:00,2.00,"2,500.00"
50%,"1,500.50",2026-03-29 00:00:00,3.00,"15,000.00"
75%,"1,750.25",2026-05-14 00:00:00,4.00,"28,000.00"
max,"2,000.00",2026-06-29 00:00:00,5.00,"65,000.00"
std,288.82,NaN,1.42,"23,908.79"


**Inspection notes.** 1,000 rows × 7 columns. `order_date` parsed to datetime (range 2026-01-01 → 2026-06-29); (0 nulls, 0 duplicates). Categories: 5 branches, 5 products, channels In-store, Online, Phone order.

## Task 2 · Derived columns

In [2]:
df["revenue"] = df["quantity"] * df["unit_price"]
df["month"] = df["order_date"].dt.month_name()
df["weekday"] = df["order_date"].dt.day_name()
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5
df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")
df[["order_id","revenue","month","weekday","is_weekend","price_band"]].head()

,order_id,revenue,month,weekday,is_weekend,price_band
0,1001,45000,June,Wednesday,False,Value
1,1002,260000,February,Monday,False,Premium
2,1003,140000,January,Monday,False,Premium
3,1004,12500,April,Sunday,True,Value
4,1005,7500,March,Saturday,True,Value


`revenue = quantity × unit_price`; `price_band` uses `unit_price ≥ KES 28,000` → Premium (so Tablet at exactly KES 28,000 is Premium; Laptop is Premium; Phone, Headphones and Charger are Value).

## Task 3 · Branch ranking: revenue vs orders

In [3]:
b = df.groupby("branch").agg(total_revenue=("revenue","sum"), orders=("order_id","count"))
b["avg_order_value"] = b.total_revenue/b.orders
b["rank_revenue"] = b.total_revenue.rank(ascending=False).astype(int)
b["rank_orders"] = b.orders.rank(ascending=False, method="min").astype(int)
b.sort_values("total_revenue", ascending=False)

,total_revenue,orders,avg_order_value,rank_revenue,rank_orders
branch,,,,,
Nairobi,26106100,376,"69,431.12",1,1
Mombasa,14003700,186,"75,288.71",2,2
Nakuru,11970400,162,"73,891.36",3,3
Kisumu,10868100,159,"68,352.83",4,4
Eldoret,8844200,117,"75,591.45",5,5


**Answer.** By revenue: Nairobi > Mombasa > Nakuru > Kisumu > Eldoret. By orders: Nairobi > Mombasa > Nakuru > Kisumu > Eldoret. The two rankings are identical: revenue = orders × average order value, and every branch sells the same product mix at similar prices, so AOV is almost flat (KES 68,353 to KES 75,591) and order volume alone decides the order. The gaps in share are small, e.g. Nairobi has 37.6% of orders but 36.4% of revenue.

## Task 4 · Top three products

In [4]:
p = df.groupby("product").agg(revenue=("revenue","sum"), units=("quantity","sum"))
print("By revenue:"); display(p.nlargest(3,"revenue"))
print("By units:");   display(p.nlargest(3,"units"))

By revenue:


,revenue,units
product,,
Laptop,41535000,639
Tablet,19348000,691
Phone,8880000,592


By units:


,revenue,units
product,,
Tablet,19348000,691
Laptop,41535000,639
Phone,8880000,592


**Answer.** Top by revenue: Laptop (KES 41,535,000), Tablet (KES 19,348,000), Phone (KES 8,880,000). Top by units: Tablet (691), Laptop (639), Phone (592). The two lists contain the same three products but in a different order. Laptop ranks first on revenue (unit price KES 65,000, 639 units) while Tablet ranks first on units (691) at only KES 28,000 each, so Laptop earns 2.1× the revenue of Tablet from fewer units. Revenue reflects price × quantity; unit counts are almost level across products because each is picked with equal probability.

## Task 5 · Weekend vs weekday

In [5]:
w = df.groupby("is_weekend").agg(orders=("order_id","count"), total_revenue=("revenue","sum"), avg_order_revenue=("revenue","mean"))
w.index = w.index.map({True:"Weekend", False:"Weekday"}); w

,orders,total_revenue,avg_order_revenue
is_weekend,,,
Weekday,723,51856200,"71,723.65"
Weekend,277,19936300,"71,972.20"


**Answer.** Weekdays: 723 orders, KES 51,856,200, avg KES 71,724. Weekends: 277 orders, KES 19,936,300, avg KES 71,972. Weekends cover only 2 of 7 days, so compare the average order value, not totals: weekend orders are larger on average (+0.3%).

## Task 6 · Best and worst month

In [6]:
m = df.groupby(df.order_date.dt.month).revenue.sum()
m.index = [pd.Timestamp(2026,i,1).month_name() for i in m.index]
best, worst = m.idxmax(), m.idxmin()
print(m); print(f"best={best} worst={worst} difference={(m[best]-m[worst])/m[worst]*100:.1f}% (best vs worst)")

January     11786600
February    11772000
March       12396800
April       13518300
May         11415400
June        10903400
Name: revenue, dtype: int64
best=April worst=June difference=24.0% (best vs worst)


**Answer.** Best: **April** (KES 13,518,300); worst: **June** (KES 10,903,400). Best is **24.0%** higher than worst, i.e. worst is 19.3% below best.

## Task 7 · Premium online orders, Nairobi & Mombasa

In [7]:
sel = df[(df.price_band=="Premium") & (df.channel=="Online") & (df.branch.isin(["Nairobi","Mombasa"]))]
print(len(sel), "orders | revenue KES", f"{sel.revenue.sum():,.0f}")
sel.to_csv("outputs/premium_online_coast_capital.csv", index=False)

83 orders | revenue KES 10,218,000


**Answer.** **83** orders, total revenue **KES 10,218,000**. Exported to `outputs/premium_online_coast_capital.csv` (`index=False`).

## Task 8 · Excel reconciliation (`excel/duka_branch_summary.xlsx`)

In [8]:
xl = pd.read_csv("excel/sumifs_recalc_check.csv").set_index("branch")      # SUMIFS/COUNTIFS values recalculated from the workbook
cmp = b[["total_revenue","orders"]].join(xl)
cmp["match"] = np.isclose(cmp.total_revenue, cmp.sumifs_revenue) & (cmp.orders == cmp.countifs_orders)
raw = pd.read_csv("excel/pivot_recalc_check.csv", header=None)         # values of the real PivotTable in the workbook
hdr_i = raw.index[raw.iloc[:,0].astype(str).str.contains("branch", case=False)][-1]
pv = raw.iloc[hdr_i+1:].copy(); pv.columns = ["branch"]+list(raw.iloc[hdr_i,1:]); pv = pv.set_index("branch")
pd_pivot = df.pivot_table(index="branch", columns="channel", values="revenue", aggfunc="sum")
ok = np.allclose(pv.loc[pd_pivot.index, pd_pivot.columns].astype(float), pd_pivot)
tot = pv.loc[pv.index.str.contains("Total", case=False)].iloc[0].astype(float).iloc[-1]
print("SUMIFS = pandas for every branch:", cmp.match.all(), "| grand total:", f"{xl.sumifs_revenue.sum():,.0f} vs {df.revenue.sum():,.0f}")
print("PivotTable branch×channel = pandas pivot:", ok, "| pivot grand total:", f"{tot:,.0f}")
cmp

SUMIFS = pandas for every branch: True | grand total: 71,792,500 vs 71,792,500
PivotTable branch×channel = pandas pivot: True | pivot grand total: 71,792,500


,total_revenue,orders,sumifs_revenue,countifs_orders,match
branch,,,,,
Eldoret,8844200,117,8844200,117,True
Kisumu,10868100,159,10868100,159,True
Mombasa,14003700,186,14003700,186,True
Nairobi,26106100,376,26106100,376,True
Nakuru,11970400,162,11970400,162,True


**Answer.** The workbook has `Branch_Summary` (SUMIFS revenue + COUNTIFS orders + XLOOKUP region from the `Regions` sheet) and `Pivot_Branch_Channel` (a real PivotTable: branch rows × channel columns, sum of revenue). The check above shows every total equals pandas.